In [1]:
# inlegalbert_dual_kg_rag_rrc.py  (DUAL KNOWLEDGE GRAPH RAG)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#   +
#   DUAL Knowledge Graphs:
#       G_all  : Global KG  (all training sentences, full label coverage)
#       G_min  : Minority KG (rare-class sentences only, fine-grained patterns)
#   +
#   Minority-Aware Retrieval: score = sim + β * rarity(node)
#   +
#   Class-Aware Top-K: K1 high-sim nodes  +  K2 forced minority nodes
#   +
#   Dynamic Fusion: α * H_all + (1-α) * H_min  (α conditioned on entropy)
#   +
#   Uncertainty-triggered KG path, Graph Attention Fusion, re-CRF
#
# NEW vs KG-RAG v1 (inlegalbert_kg_rag_rrc.py):
#   1. DualKnowledgeGraph: builds G_all (full) + G_min (minority-only) separately.
#   2. MinorityAwareRetriever:
#        - Rarity bonus:  score(node) = sim + β * rarity(node)
#        - Class-aware Top-K: K1 sim-based + K2 forced minority quota
#   3. DualKGRetriever: queries both G_all and G_min, returns (H_all, H_min).
#   4. DynamicFusion: entropy-conditioned α weighting of H_all vs H_min.
#   5. DualKGAugmentedModel: integrates dual retrieval + dynamic fusion.
#   6. All anti-overfitting settings from v1 retained.

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_dual_kg_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# BERT freeze / layer-wise LR decay
BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

# Sentence-level BiLSTM
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# Multi-Head Attention Pooling
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1

# Context-enrichment BiLSTM
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

# Auxiliary loss
AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# Early stopping
ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── DUAL KG-RAG specific ───────────────────────────────────
KG_TOP_K            = 3      # top-K subgraphs from Global KG
KG_TOP_NODES        = 5      # top similarity nodes per subgraph (K1)
KG_MIN_NODES        = 3      # forced minority nodes per retrieval (K2)
KG_HOP              = 1
UNCERTAINTY_THRESH  = 0.7
RARE_ALWAYS_KG      = True
KG_FUSION_DIM       = 256    # == sent_out_dim

# Minority-Aware Retrieval
RARITY_BETA         = 0.3    # β in: score = sim + β * rarity(node)

# Dynamic Fusion (α * H_all + (1-α) * H_min)
ALPHA_LOW_ENTROPY   = 0.7    # confident → trust global more
ALPHA_HIGH_ENTROPY  = 0.4    # uncertain → trust minority more

# RST edge similarity thresholds
RST_INTRA_THRESH    = 0.6
RST_CROSS_THRESH    = 0.5

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2        # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = self.sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = self.sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2          # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        """Returns sentence-level embeddings: (B, T, sent_out_dim)."""
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def get_emissions(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ):
        """Returns (sent_vecs, ctx_out, emissions) for external use."""
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH  (shared base class)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    Stores sentence embeddings per label. Shared by Global and Minority variants.
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, name="global"):
        self.emb_dim     = emb_dim
        self.name        = name
        self.nodes       = defaultdict(list)
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        self._stacked    = {}

    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def build_edges(self,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        print(f"  Building {self.name} KG edges ...")
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs      = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat   = torch.mm(embs_norm, embs_norm.T)

            for i in range(N - 1):
                w = float(sim_mat[i, i + 1].item())
                self.intra_edges[lid].append((i, i + 1, max(0.0, w)))

            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    w = float(sims[j].item())
                    self.intra_edges[lid].append((i, j, w))
                    sims[j] = -1
                    count += 1

            self._stacked[lid] = embs

        label_ids   = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb    = label_ids[a], label_ids[b]
                embs_a    = self._get_stacked(la)
                embs_b    = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                na_norm   = F.normalize(embs_a, dim=-1)
                nb_norm   = F.normalize(embs_b, dim=-1)
                sim_mat   = torch.mm(na_norm, nb_norm.T)
                high      = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        n_nodes = sum(len(v) for v in self.nodes.values())
        print(f"  [{self.name}] KG: {n_nodes} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        data = {
            "name": self.name,
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  [{self.name}] KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        with open(path) as f:
            data = json.load(f)
        kg = cls(emb_dim=emb_dim, name=data.get("name", "global"))
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]),
                                      "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  [{kg.name}] KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# DUAL KNOWLEDGE GRAPH  (G_all + G_min)
# ═══════════════════════════════════════════════════════════
class DualKnowledgeGraph:
    """
    Maintains two separate KGs:
      G_all : Global KG — all training sentences, full label coverage
      G_min : Minority KG — rare-class sentences only
    """
    def __init__(self, rare_ids: list, emb_dim=KG_FUSION_DIM):
        self.rare_ids  = set(rare_ids)
        self.g_all     = KnowledgeGraph(emb_dim=emb_dim, name="global")
        self.g_min     = KnowledgeGraph(emb_dim=emb_dim, name="minority")

    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        """Route every node to G_all; rare-class nodes also go to G_min."""
        self.g_all.add_nodes(embeddings, label_ids, texts)

        # Filter for minority only
        min_embs, min_ids, min_texts = [], [], []
        for k, (emb, lid) in enumerate(zip(embeddings, label_ids)):
            if lid in self.rare_ids:
                min_embs.append(emb)
                min_ids.append(lid)
                if texts is not None:
                    min_texts.append(texts[k])

        if min_embs:
            min_emb_tensor = torch.stack(min_embs)
            self.g_min.add_nodes(
                min_emb_tensor, min_ids,
                min_texts if texts else None
            )

    def build_edges(self):
        self.g_all.build_edges()
        self.g_min.build_edges(
            intra_thresh=RST_INTRA_THRESH - 0.1,   # slightly looser for minority
            cross_thresh=RST_CROSS_THRESH - 0.1,
        )

    def save(self, dir_path):
        self.g_all.save(os.path.join(dir_path, "kg_global.json"))
        self.g_min.save(os.path.join(dir_path, "kg_minority.json"))

    @classmethod
    def load(cls, dir_path, rare_ids, emb_dim=KG_FUSION_DIM):
        dual = cls(rare_ids=rare_ids, emb_dim=emb_dim)
        dual.g_all = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_global.json"), emb_dim=emb_dim
        )
        dual.g_min = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_minority.json"), emb_dim=emb_dim
        )
        return dual


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor) -> torch.Tensor:
        return self.entropy(logits) > self.threshold

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# MINORITY-AWARE RETRIEVER  (for a single KG)
# ═══════════════════════════════════════════════════════════
class MinorityAwareRetriever:
    """
    Retrieval from a single KG with rarity bonus:
        score(node) = sim(query, node) + β * rarity(node)
    where rarity(node) = 1 / log(freq_class(node) + 1).

    Also supports Class-Aware Top-K:
        K1 nodes selected by boosted score
        K2 forced minority nodes (quota)
    """
    def __init__(self, kg: KnowledgeGraph, label_freqs: dict,
                 rare_ids: list,
                 top_k=KG_TOP_K,
                 top_nodes_k1=KG_TOP_NODES,
                 top_nodes_k2=KG_MIN_NODES,
                 hop=KG_HOP,
                 beta=RARITY_BETA):
        self.kg          = kg
        self.rare_ids    = set(rare_ids)
        self.top_k       = top_k
        self.top_nodes_k1 = top_nodes_k1
        self.top_nodes_k2 = top_nodes_k2
        self.hop         = hop
        self.beta        = beta

        # Precompute rarity score per label_id
        # rarity(lid) = 1 / log(freq + 1 + eps)
        self.rarity = {}
        for i in range(NUM_LABELS):
            freq = label_freqs.get(id2label[i], 0.0)
            self.rarity[i] = 1.0 / (math.log(freq + 1.0 + 1e-6))
        # Normalise rarity to [0,1]
        max_rar = max(self.rarity.values()) or 1.0
        self.rarity = {k: v / max_rar for k, v in self.rarity.items()}

    def retrieve(self, h_i: torch.Tensor) -> list:
        """
        h_i: (D,) CPU.
        Returns list of (emb: Tensor(D,), weight: float).
        """
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)   # (1, D)

        # ── Step 1: score each subgraph with rarity bonus ─
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            base_sims = torch.mv(embs_norm, h_norm.squeeze(0))   # (N,)
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            boosted = base_sims + rarity_bonus
            subgraph_scores[lid] = float(boosted.max().item())

        # ── Step 2: top-K subgraphs ───────────────────────
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []

        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            base_sims = torch.mv(embs_norm, h_norm.squeeze(0))  # (N,)
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            boosted = base_sims + rarity_bonus

            # ── Class-Aware Top-K ─────────────────────────
            # K1: top by boosted score
            k1 = min(self.top_nodes_k1, embs.shape[0])
            k1_idx  = boosted.topk(k1).indices.tolist()
            k1_sims = base_sims[k1_idx].tolist()

            seed_set = set(k1_idx)

            # K2: forced minority quota (only if this label is rare)
            if lid in self.rare_ids:
                k2 = min(self.top_nodes_k2, embs.shape[0])
                # take top-K2 by raw sim (no need for extra boost here)
                k2_idx = base_sims.topk(k2).indices.tolist()
                for idx in k2_idx:
                    if idx not in seed_set:
                        seed_set.add(idx)
                        k1_sims.append(float(base_sims[idx].item()))
                        k1_idx.append(idx)

            # ── 1-hop expansion ───────────────────────────
            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        cross_embs = self.kg._get_stacked(lb)
                        if cross_embs is not None and nj < cross_embs.shape[0]:
                            results.append((cross_embs[nj], w))
                    elif lb == lid and nj in seed_set:
                        cross_embs = self.kg._get_stacked(la)
                        if cross_embs is not None and ni < cross_embs.shape[0]:
                            results.append((cross_embs[ni], w))

            for idx, sim in zip(k1_idx, k1_sims):
                results.append((embs[idx], float(sim)))

        return results


# ═══════════════════════════════════════════════════════════
# DUAL KG RETRIEVER  (queries G_all + G_min)
# ═══════════════════════════════════════════════════════════
class DualKGRetriever:
    """
    Wraps two MinorityAwareRetrievers (one for G_all, one for G_min).
    Returns (neighbours_all, neighbours_min) tuples.
    """
    def __init__(self, dual_kg: DualKnowledgeGraph, label_freqs: dict,
                 rare_ids: list):
        self.retriever_all = MinorityAwareRetriever(
            kg=dual_kg.g_all, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA,
        )
        self.retriever_min = MinorityAwareRetriever(
            kg=dual_kg.g_min, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA * 1.5,
            # stronger rarity bias inside Minority KG
        )

    def retrieve(self, h_i: torch.Tensor):
        """Returns (H_all, H_min) each as list of (Tensor(D,), float)."""
        h_all = self.retriever_all.retrieve(h_i)
        h_min = self.retriever_min.retrieve(h_i)
        return h_all, h_min


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    v_i = Σ α_j * emb_j,  α_j ∝ sim(h_i, emb_j) × edge_weight_j
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor,
                neighbours: list, device=None) -> torch.Tensor:
        """
        h_i        : (D,) on device
        neighbours : list of (Tensor(D,), float weight)  — may be on CPU
        Returns    : v_i (D,) on device
        """
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)

        q = self.proj_q(h_i.unsqueeze(0))
        k = self.proj_k(embs)
        dot   = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)

        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


# ═══════════════════════════════════════════════════════════
# DYNAMIC FUSION  (α * V_all + (1-α) * V_min)
# ═══════════════════════════════════════════════════════════
class DynamicFusion(nn.Module):
    """
    Entropy-conditioned weighting of global vs minority graph vectors.
    Additionally learns a scalar gate per position.
    """
    def __init__(self, emb_dim=KG_FUSION_DIM):
        super().__init__()
        # Learnable gate: input = concat(h_i, v_all, v_min) → scalar α
        self.gate = nn.Sequential(
            nn.Linear(emb_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, h_i: torch.Tensor,
                v_all: torch.Tensor, v_min: torch.Tensor,
                entropy: float) -> torch.Tensor:
        """
        h_i, v_all, v_min : (D,) all on same device.
        entropy            : normalised entropy ∈ [0,1]
        Returns fused (D,).
        """
        # Fixed entropy-based α
        alpha_fixed = ALPHA_HIGH_ENTROPY if entropy > UNCERTAINTY_THRESH \
                      else ALPHA_LOW_ENTROPY

        # Learnable gate (refines fixed α)
        gate_input = torch.cat([h_i, v_all, v_min], dim=-1).unsqueeze(0)
        gate_val   = self.gate(gate_input).squeeze()   # scalar

        # Combine: use gate as a blend of fixed and learned
        alpha = 0.5 * alpha_fixed + 0.5 * gate_val.item()
        alpha = max(0.2, min(0.8, alpha))              # clip to [0.2, 0.8]

        return alpha * v_all + (1.0 - alpha) * v_min


# ═══════════════════════════════════════════════════════════
# DUAL KG AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class DualKGAugmentedModel(nn.Module):
    """
    Extends base model with Dual KG retrieval + dynamic fusion.

    Forward pass:
      1. base.get_emissions  → sent_vecs, ctx_out, base_emissions
      2. Uncertainty check per sentence
      3. For uncertain / rare sentences:
           a. Retrieve (H_all, H_min) from DualKGRetriever
           b. GraphAttentionFusion(h_i, H_all) → V_all
           c. GraphAttentionFusion(h_i, H_min) → V_min
           d. DynamicFusion(h_i, V_all, V_min, entropy) → V_i
           e. h* = h_i + V_i  (residual)
      4. fused_sent → ctx_bilstm → fusion_classifier → fusion_crf
      5. combined_emissions = (base + fused) / 2
    """
    def __init__(
        self,
        base_model:   InLegalBERT_BiLSTM_MHA_CRF,
        dual_kg:      DualKnowledgeGraph,
        dual_retriever: DualKGRetriever,
        rare_ids:     list,
    ):
        super().__init__()
        self.base          = base_model
        self.dual_kg       = dual_kg
        self.retriever     = dual_retriever
        self.rare_ids      = set(rare_ids)
        self.uncertainty   = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        # One GAT for Global, one for Minority
        self.gat_all = GraphAttentionFusion(emb_dim=sent_dim)
        self.gat_min = GraphAttentionFusion(emb_dim=sent_dim)

        # Dynamic fusion
        self.dyn_fusion = DynamicFusion(emb_dim=sent_dim)

        # Projection: fused sent_dim → ctx_dim for classifier reuse
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _fuse_batch(self, sent_vecs, emissions, lengths, device):
        """
        Returns fused_sent_vecs (B, T, sent_dim).
        For uncertain/rare positions: h* = h_i + DynamicFusion(V_all, V_min)
        """
        B, T, sent_dim = sent_vecs.shape
        fused = sent_vecs.clone()

        entropy_map    = self.uncertainty.entropy(emissions)   # (B, T) ∈ [0,1]
        uncertain_mask = entropy_map > UNCERTAINTY_THRESH       # bool (B, T)
        top_labels     = self.uncertainty.top_label(emissions) # (B, T)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids

                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue

                h_i_cpu = sent_vecs[b, t].detach().cpu()
                h_all, h_min = self.retriever.retrieve(h_i_cpu)

                if not h_all and not h_min:
                    continue

                h_i_gpu = sent_vecs[b, t]   # keep on GPU for grad
                ent_val = float(entropy_map[b, t].item())

                v_all = self.gat_all(h_i_gpu, h_all, device=device) \
                        if h_all else torch.zeros(sent_dim, device=device)
                v_min = self.gat_min(h_i_gpu, h_min, device=device) \
                        if h_min else torch.zeros(sent_dim, device=device)

                # Dynamic fusion: α * V_all + (1-α) * V_min
                v_i = self.dyn_fusion(h_i_gpu, v_all, v_min, ent_val)

                fused[b, t] = h_i_gpu + v_i   # residual

        return fused

    def forward(
        self,
        input_ids, attention_mask, token_type_ids,
        labels=None, lengths=None,
    ):
        device = input_ids.device

        # ── Base model ────────────────────────────────────
        sent_vecs, _, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # ── Dual KG fusion ────────────────────────────────
        fused_sent = self._fuse_batch(sent_vecs, base_emissions, lengths, device)

        # ── Context BiLSTM on fused sent ──────────────────
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # ── Fusion classifier ─────────────────────────────
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        # ── CRF mask ──────────────────────────────────────
        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2],
                              dtype=torch.bool, device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss  = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )
            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (Phase A)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []

        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                labels    = labels.to(self.device)
                lengths   = lengths.to(self.device)
                loss, _   = self.model(input_ids, attn, ttype,
                                       labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                lengths   = lengths.to(self.device)
                decoded, _ = self.model(input_ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)

        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper       = EarlyStopping()
        history             = []
        best_f1, best_state = -1.0, None
        total_start         = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        hist_df    = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))

        return hist_df, total_time


# ═══════════════════════════════════════════════════════════
# DUAL KG BUILDER  (after Phase A)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_dual_knowledge_graph(base_model, train_docs, tokenizer,
                                rare_ids, device=DEVICE) -> DualKnowledgeGraph:
    print("\n🔨 Building Dual Knowledge Graph (G_all + G_min) ...")
    base_model.eval()
    base_model.to(device)

    dual_kg = DualKnowledgeGraph(rare_ids=rare_ids,
                                  emb_dim=base_model.sent_out_dim)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader        = DataLoader(dummy_dataset, batch_size=1,
                               shuffle=False, collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)

        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n         = int(lengths[0].item())
        embs      = sent_vecs[:n].cpu()
        lab_ids   = labels[0, :n].tolist()

        dual_kg.add_nodes(embs, lab_ids)

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(loader)} docs")

    dual_kg.build_edges()
    dual_kg.save(OUT_DIR)

    n_all = sum(len(v) for v in dual_kg.g_all.nodes.values())
    n_min = sum(len(v) for v in dual_kg.g_min.nodes.values())
    print(f"  G_all nodes: {n_all} | G_min nodes: {n_min}")
    return dual_kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (Phase B)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: DualKGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_all.parameters()) +
            list(self.model.gat_min.parameters()) +
            list(self.model.dyn_fusion.parameters()) +
            list(self.model.fusion_proj.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters())
        )
        base_trainable = [p for p in self.model.base.parameters()
                          if p.requires_grad]

        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)

                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)

        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper       = EarlyStopping(patience=5)
        history             = []
        best_f1, best_state = -1.0, None
        total_start         = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item()
                n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[DualKG] Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1:  {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "dual_kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best DualKG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  DualKG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "dual_kg_history.csv"), index=False
        )

        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "dual_kg_model.bin"))
            print(f"\n✔ Best DualKG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (Dual KG-RAG)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    labels = LABELS
    f1s    = [per_class_metrics[l]["f1"] for l in labels]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in labels]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (Dual KG-RAG)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            label="Train Loss", marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1", marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → Dual KG-RAG)"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None,
                        total_trainable=None):
    rows = [
        ("Accuracy",            "accuracy"),
        ("Macro-F1",            "macro_f1"),
        ("Micro-F1",            "micro_f1"),
        ("Weighted-F1",         "weighted_f1"),
        ("Rare / Minority F1",  "rare_f1"),
        ("Macro-Precision",     "macro_precision"),
        ("Macro-Recall",        "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + Dual KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  Dual KG-RAG\n")
    print("Dual KG = G_all (Global) + G_min (Minority-only)")
    print("Retrieval = Minority-Aware (rarity bonus β) + Class-Aware Top-K (K1+K2)")
    print("Fusion = Dynamic α (entropy-conditioned + learned gate)\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ════════════════════════════════════════════════════
    # PHASE A→B: Build Dual KG
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Dual Knowledge Graph")
    print("=" * 60)

    kg_global_path   = os.path.join(OUT_DIR, "kg_global.json")
    kg_minority_path = os.path.join(OUT_DIR, "kg_minority.json")

    if os.path.exists(kg_global_path) and os.path.exists(kg_minority_path):
        print("  Found existing Dual KG, loading ...")
        dual_kg = DualKnowledgeGraph.load(OUT_DIR, rare_ids=rare_ids,
                                           emb_dim=base_model.sent_out_dim)
    else:
        dual_kg = build_dual_knowledge_graph(
            base_model, train_docs, tokenizer, rare_ids, DEVICE
        )

    # ════════════════════════════════════════════════════
    # PHASE B: Dual KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: Dual KG-Augmented Fine-Tuning")
    print("=" * 60)

    dual_retriever = DualKGRetriever(
        dual_kg=dual_kg,
        label_freqs=label_freqs,
        rare_ids=rare_ids,
    )

    kg_model = DualKGAugmentedModel(
        base_model     = base_model,
        dual_kg        = dual_kg,
        dual_retriever = dual_retriever,
        rare_ids       = rare_ids,
    )

    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )

    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                       split_name="dev",
                                       measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + Dual KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                        split_name="test",
                                        measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + Dual KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + Dual KG-RAG",
        "dual_kg_config": {
            "top_k": KG_TOP_K,
            "top_nodes_k1": KG_TOP_NODES,
            "top_nodes_k2": KG_MIN_NODES,
            "hop": KG_HOP,
            "uncertainty_thresh": UNCERTAINTY_THRESH,
            "rare_always_kg": RARE_ALWAYS_KG,
            "rarity_beta": RARITY_BETA,
            "alpha_low_entropy": ALPHA_LOW_ENTROPY,
            "alpha_high_entropy": ALPHA_HIGH_ENTROPY,
            "intra_thresh": RST_INTRA_THRESH,
            "cross_thresh": RST_CROSS_THRESH,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  Dual KG-RAG

Dual KG = G_all (Global) + G_min (Minority-only)
Retrieval = Minority-Aware (rarity bonus β) + Class-Aware Top-K (K1+K2)
Fusion = Dynamic α (entropy-conditioned + learned gate)

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RAR

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 37.9s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 36.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 40.7s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 37.1s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 38.3s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val